# Omnitech.SignalCortex — Kaggle Training

Train the multi-scale LSTM model on Kaggle T4 GPU.

## Setup
1. Upload `signal-cortex-data` dataset (Parquet files)
2. Upload `signal-cortex-code` dataset (project code)
3. Enable GPU accelerator (T4)

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q scikit-learn pandas pyarrow pyyaml tqdm matplotlib seaborn tensorboard

In [ ]:
import sys, os, shutil

# Kaggle dataset paths
CODE_DATASET = "/kaggle/input/signal-cortex-code"
DATA_DATASET = "/kaggle/input/signal-cortex-data"
WORK_DIR = "/kaggle/working/SignalCortex"

# Copy code to working dir (input is read-only)
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(CODE_DATASET, WORK_DIR)

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print(f"Working dir: {os.getcwd()}")
print(f"Parquet dir: {DATA_DATASET}")
print(f"GPU: {os.popen('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader').read().strip()}")

In [ ]:
# Verify data
import pandas as pd
for f in ['features_5m.parquet', 'features_15m.parquet', 'features_1h.parquet']:
    path = os.path.join(DATA_DATASET, f)
    df = pd.read_parquet(path)
    print(f"{f}: {len(df):,} rows, {df['pair_name'].nunique()} pairs")

In [ ]:
# Train — Large model with ATR-based labels
!python main.py train \
    --config configs/experiments/multi_pair_v2_large.yaml \
    --parquet {DATA_DATASET}

In [ ]:
# Resume training if session was interrupted
# Uncomment and update the checkpoint path
# !python main.py train \
#     --config configs/experiments/multi_pair_v2_large.yaml \
#     --parquet {DATA_DATASET} \
#     --resume outputs/multi_pair_v2_large/best_model.pt

In [ ]:
# Backtest on ETHUSDT (unseen pair)
!python main.py backtest \
    --config configs/experiments/multi_pair_v2_large.yaml \
    --checkpoint outputs/multi_pair_v2_large/best_model.pt \
    --pair ETHUSDT \
    --parquet {DATA_DATASET}

In [ ]:
# Copy outputs to Kaggle output dir for download
import shutil
output_src = os.path.join(WORK_DIR, "outputs")
output_dst = "/kaggle/working/outputs"
if os.path.exists(output_src):
    shutil.copytree(output_src, output_dst, dirs_exist_ok=True)
    print(f"Outputs copied to {output_dst}")
    for root, dirs, files in os.walk(output_dst):
        for f in files:
            path = os.path.join(root, f)
            size = os.path.getsize(path) / 1024 / 1024
            print(f"  {path} ({size:.1f} MB)")